# Job ETL

Neste notebook, é aplicado o Job ETL. Ele é um processo que extrai, transforma e carrega dados de diferentes fontes para um destino central. Ele organiza e prepara os dados para que possam ser usados em análises ou relatórios. No caso desse projeto, a fonte será do arquivo Complete_Pokedex_V1.1.csv e o resultado será utilizado na camada gold.

## 1. Análise do Dataset Bruto (Antes do ETL)

Nesta etapa, fazemos uma análise inicial do nosso dataset bruto (`raw`). Com isso, o objetivo é registrar o estado original dos dados e mostrar a quantidade de linhas, de colunas e a presença de valores nulos, antes de qualquer transformação.

In [1]:
import pandas as pd

# --- CONFIGURAÇÕES DE EXIBIÇÃO DO PANDAS ---
# Remove o limite de linhas e colunas a serem exibidas
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)


#========= EXTRAÇÃO ==========
dataFrame = pd.read_csv('../raw/Complete_Pokedex_V1.1.csv') 


#========= DIAGNÓSTICO DO DATASET ==========
print("--- DIAGNÓSTICO COMPLETO DO DATASET BRUTO (RAW) ---")

# 1. Dimensões do DataFrame
print("\n--- 1. DIMENSÕES DO DATASET ---")
linhas, colunas = dataFrame.shape
print(f"Quantidade de Linhas: {linhas}")
print(f"Quantidade de Colunas: {colunas}")


# 2. Verificação de Dados Duplicados
print("\n--- 2. VERIFICAÇÃO DE DUPLICATAS ---")
duplicatas = dataFrame.duplicated().sum()
print(f"Quantidade de Linhas Duplicadas: {duplicatas}")

# 3. Análise de Valores Nulos
print("\n--- 3. ANÁLISE DE VALORES NULOS ---")
nulos_por_coluna = dataFrame.isnull().sum()
colunas_com_nulos = nulos_por_coluna[nulos_por_coluna > 0].sort_values(ascending=False)
if not colunas_com_nulos.empty:
    print("Contagem de valores nulos (apenas colunas com dados faltantes):")
    print(colunas_com_nulos.to_string())
else:
    print("Não há colunas com valores nulos no dataset.")

# 4. Análise de Colunas Categóricas
print("\n--- 4. ANÁLISE DE CATEGORIAS IMPORTANTES ---")
print(f"Quantidade de Gerações Únicas: {dataFrame['generation'].nunique()}")
print(f"Quantidade de Tipos Primários Únicos: {dataFrame['type_1'].nunique()}")
print("\nContagem de Pokémon por Geração:")
print(dataFrame['generation'].value_counts().sort_index().to_string())





--- DIAGNÓSTICO COMPLETO DO DATASET BRUTO (RAW) ---

--- 1. DIMENSÕES DO DATASET ---
Quantidade de Linhas: 1118
Quantidade de Colunas: 63

--- 2. VERIFICAÇÃO DE DUPLICATAS ---
Quantidade de Linhas Duplicadas: 0

--- 3. ANÁLISE DE VALORES NULOS ---
Contagem de valores nulos (apenas colunas com dados faltantes):
egg_group_2     799
ability_3       580
evolves_from    568
type_2          521
ability_2       257

--- 4. ANÁLISE DE CATEGORIAS IMPORTANTES ---
Quantidade de Gerações Únicas: 8
Quantidade de Tipos Primários Únicos: 18

Contagem de Pokémon por Geração:
generation
1    151
2    100
3    138
4    118
5    165
6    141
7    149
8    156


## 2. Job ETL: Extração, Transformação e Carregamento


Aqui executamos o processo de transformação (ETL). Esta célula utiliza o `dataFrame` carregado na etapa anterior, aplica as regras de limpeza e, por fim, salva o resultado em um novo arquivo CSV na camada `Silver`

In [ ]:
import pandas as pd

#=========EXTRAÇÃO==========
dataFrame = pd.read_csv('../raw/Complete_Pokedex_V1.1.csv') 

#=========TRANSFORMAÇÃO==========

colunas_para_apagar = [
    'ability_1', 
    'ability_2', 
    'ability_3',
    'number_pokemon_with_typing', 
    'primary_color'
]

dataFrame = dataFrame.drop(columns=colunas_para_apagar) 
# dropna exclui as linhas nulas e o subset especifica de quais colunas será
dataFrame = dataFrame.dropna(subset=["type_2", "evolves_from"]) 

# Apaga colunas
dataFrame = dataFrame.drop(columns=["mean", "standard_deviation", "exp_to_level_100", "can_evolve", "final_evolution"]) 

#=========LOAD==========

# salva em novo csv sem o indice automatico
dataFrame.to_csv("Complete_Pokedex-Tratada.csv", index=False)

print("\n Job ETL concluído!")


✅ Job ETL concluído!


## 3. Análise do Dataset Tratado (após ETL)

Nesta etapa, carregamos o dataset que foi tratado e salvo na camada 'silver' (`Complete_Pokedex-Tratada.csv`). Realizamos um diagnóstico semelhante ao inicial para verificar o resultado das transformações e garantir a qualidade dos dados que serão usados nas análises visuais.

In [3]:
import pandas as pd

# --- CONFIGURAÇÕES DE EXIBIÇÃO DO PANDAS ---
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)


#========= EXTRAÇÃO DO ARQUIVO TRATADO ==========
try:
    df_tratado = pd.read_csv('Complete_Pokedex-Tratada.csv')
    print("Arquivo 'Complete_Pokedex-Tratada.csv' carregado com sucesso!")
except FileNotFoundError:
    print("ERRO: O arquivo 'Complete_Pokedex-Tratada.csv' não foi encontrado.")
    print("Por favor, execute a Célula 2 (Job ETL) primeiro para gerar o arquivo.")


#========= DIAGNÓSTICO DO DATASET ==========
if 'df_tratado' in locals():
    print("\n--- DIAGNÓSTICO COMPLETO DO DATASET TRATADO (SILVER) ---")

    # 1. Dimensões do DataFrame
    print("\n--- 1. DIMENSÕES DO DATASET ---")
    linhas, colunas = df_tratado.shape
    print(f"Quantidade de Linhas: {linhas}")
    print(f"Quantidade de Colunas: {colunas}")

    # 2. Verificação de Dados Duplicados
    print("\n--- 2. VERIFICAÇÃO DE DUPLICATAS ---")
    duplicatas = df_tratado.duplicated().sum()
    print(f"Quantidade de Linhas Duplicadas: {duplicatas}")

    # 3. Análise de Valores Nulos
    print("\n--- 3. ANÁLISE DE VALORES NULOS ---")
    nulos_por_coluna = df_tratado.isnull().sum()
    colunas_com_nulos = nulos_por_coluna[nulos_por_coluna > 0]
    if not colunas_com_nulos.empty:
        print("Contagem de valores nulos (apenas colunas com dados faltantes):")
        print(colunas_com_nulos.to_string())

    # 4. Análise de Colunas Categóricas
    print("\n--- 4. ANÁLISE DE CATEGORIAS IMPORTANTES ---")
    print(f"Quantidade de Gerações Únicas: {df_tratado['generation'].nunique()}")
    print(f"Quantidade de Tipos Primários Únicos: {df_tratado['type_1'].nunique()}")
    print("\nContagem de Pokémon por Geração (após a limpeza):")
    print(df_tratado['generation'].value_counts().sort_index().to_string())

Arquivo 'Complete_Pokedex-Tratada.csv' carregado com sucesso!

--- DIAGNÓSTICO COMPLETO DO DATASET TRATADO (SILVER) ---

--- 1. DIMENSÕES DO DATASET ---
Quantidade de Linhas: 311
Quantidade de Colunas: 53

--- 2. VERIFICAÇÃO DE DUPLICATAS ---
Quantidade de Linhas Duplicadas: 0

--- 3. ANÁLISE DE VALORES NULOS ---
Contagem de valores nulos (apenas colunas com dados faltantes):
egg_group_2    201

--- 4. ANÁLISE DE CATEGORIAS IMPORTANTES ---
Quantidade de Gerações Únicas: 8
Quantidade de Tipos Primários Únicos: 18

Contagem de Pokémon por Geração (após a limpeza):
generation
1    40
2    23
3    38
4    35
5    36
6    53
7    39
8    47
